In [5]:
import os
import shutil
from datetime import datetime

backup_ts = datetime.now().strftime("%Y%m%d_%H%M%S")

source_root = "/lakehouse/default/Files"
backup_root = f"/lakehouse/default/Files/backup/before_cicd_test_{backup_ts}"

folders_to_backup = [
    "raw",
    "processed",
    "logs",
    "error"
]

os.makedirs(backup_root, exist_ok=True)

for folder in folders_to_backup:
    src = os.path.join(source_root, folder)
    dst = os.path.join(backup_root, "Files", folder)

    try:
        if os.path.exists(src):
            print(f"Backing up {src} -> {dst}")
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"Completed: {folder}")
        else:
            print(f"Skipped: {src} does not exist")

    except Exception as e:
        print(f"Failed: {src} | Error: {e}")

print(f"File backup completed at: {backup_root}")

Backing up /lakehouse/default/Files/raw -> /lakehouse/default/Files/backup/before_cicd_test_20260601_010839/Files/raw
Completed: raw
Backing up /lakehouse/default/Files/processed -> /lakehouse/default/Files/backup/before_cicd_test_20260601_010839/Files/processed
Completed: processed
Backing up /lakehouse/default/Files/logs -> /lakehouse/default/Files/backup/before_cicd_test_20260601_010839/Files/logs
Completed: logs
Skipped: /lakehouse/default/Files/error does not exist
File backup completed at: /lakehouse/default/Files/backup/before_cicd_test_20260601_010839


In [6]:
metadata_tables = [
    "audit_ingestion_run",
    "metadata_config",
    "pipeline_execution_log"
]

for table in metadata_tables:
    try:
        df = spark.table(table)
        target_path = f"{backup_root}/metadata_tables_csv/{table}"

        print(f"Exporting {table} -> {target_path}")

        df.coalesce(1) \
          .write \
          .mode("overwrite") \
          .option("header", "true") \
          .csv(target_path)

    except Exception as e:
        print(f"Failed table: {table} | Error: {e}")

print(f"Metadata table backup completed at: {backup_root}/metadata_tables_csv")

Failed table: audit_ingestion_run | Error: name 'spark' is not defined
Failed table: metadata_config | Error: name 'spark' is not defined
Failed table: pipeline_execution_log | Error: name 'spark' is not defined
Metadata table backup completed at: /lakehouse/default/Files/backup/before_cicd_test_20260601_010839/metadata_tables_csv


In [7]:
import shutil

shutil.make_archive(
    "/lakehouse/default/Files/export",
    'zip',
    "/lakehouse/default/Files/processed"
)

'/lakehouse/default/Files/export.zip'

In [9]:
import notebookutils

files = notebookutils.fs.ls("Files")
for f in files:
    print(f.name)

backup


In [10]:
import notebookutils

files = notebookutils.fs.ls("Files")
for f in files:
    print(f.name)

backup


In [12]:
import shutil
import notebookutils

# Create zip locally
shutil.make_archive(
    "/tmp/export",
    "zip",
    "/lakehouse/default/Files/backup"
)

# Copy to OneLake
notebookutils.fs.cp(
    "file:/tmp/export.zip",
    "Files/export.zip"
)

True

In [14]:
import os
import zipfile
import notebookutils

source_folder = "/lakehouse/default/Files/backup"
local_zip_path = "/tmp/backup_export.zip"
target_zip_path = "Files/backup_export.zip"

# Remove old zip if exists
if os.path.exists(local_zip_path):
    os.remove(local_zip_path)

# Create zip locally
with zipfile.ZipFile(local_zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(source_folder):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, source_folder)
            zipf.write(file_path, arcname)

print("Local zip created:", local_zip_path)
print("Zip size:", os.path.getsize(local_zip_path))

# Copy zip to OneLake Files
notebookutils.fs.cp(
    "file:" + local_zip_path,
    target_zip_path,
    True
)

print("Copied to OneLake:", target_zip_path)

# Verify in OneLake
display(notebookutils.fs.ls("Files"))

Local zip created: /tmp/backup_export.zip
Zip size: 17509250
Copied to OneLake: Files/backup_export.zip


In [15]:
import zipfile
import os

zip_file = "/lakehouse/default/Files/lakehouse_backup.zip"
source_folder = "/lakehouse/default/Files"

with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(source_folder):
        for file in files:
            if file != "lakehouse_backup.zip":
                filepath = os.path.join(root, file)
                arcname = os.path.relpath(filepath, source_folder)
                zipf.write(filepath, arcname)

print("Backup created:", zip_file)

Backup created: /lakehouse/default/Files/lakehouse_backup.zip


In [16]:
from datetime import datetime
import os
import json

backup_ts = datetime.now().strftime("%Y%m%d_%H%M%S")

base_backup_path = f"/lakehouse/default/Files/backup/lakehouse_table_backup_{backup_ts}"
data_backup_path = f"{base_backup_path}/tables_data"
schema_backup_path = f"{base_backup_path}/tables_schema"

os.makedirs(data_backup_path, exist_ok=True)
os.makedirs(schema_backup_path, exist_ok=True)

print("Backup path:", base_backup_path)

Backup path: /lakehouse/default/Files/backup/lakehouse_table_backup_20260601_015727


In [17]:
tables = spark.sql("SHOW TABLES").collect()

for t in tables:
    print(t.database, t.tableName, t.isTemporary)

NameError: name 'spark' is not defined

In [18]:
import os
import shutil
import json
from datetime import datetime

backup_ts = datetime.now().strftime("%Y%m%d_%H%M%S")

source_tables_path = "/lakehouse/default/Tables"
backup_base_path = f"/lakehouse/default/Files/backup/lakehouse_full_table_backup_{backup_ts}"

tables_data_backup_path = f"{backup_base_path}/tables_data"
tables_schema_backup_path = f"{backup_base_path}/tables_schema"

os.makedirs(tables_data_backup_path, exist_ok=True)
os.makedirs(tables_schema_backup_path, exist_ok=True)

print("Backup started:", backup_base_path)

for table_name in os.listdir(source_tables_path):
    source_table_path = os.path.join(source_tables_path, table_name)

    if os.path.isdir(source_table_path):
        target_table_path = os.path.join(tables_data_backup_path, table_name)

        print(f"Backing up table folder: {table_name}")

        shutil.copytree(
            source_table_path,
            target_table_path,
            dirs_exist_ok=True
        )

        schema_info = {
            "table_name": table_name,
            "source_path": source_table_path,
            "backup_path": target_table_path,
            "backup_timestamp": backup_ts,
            "note": "This is a Delta table folder backup including parquet data and _delta_log metadata."
        }

        schema_file_path = os.path.join(
            tables_schema_backup_path,
            f"{table_name}_backup_info.json"
        )

        with open(schema_file_path, "w", encoding="utf-8") as f:
            json.dump(schema_info, f, indent=4)

print("Backup completed successfully.")
print("Backup location:", backup_base_path)

Backup started: /lakehouse/default/Files/backup/lakehouse_full_table_backup_20260601_021744
Backing up table folder: dbo
Backup completed successfully.
Backup location: /lakehouse/default/Files/backup/lakehouse_full_table_backup_20260601_021744


In [20]:
import shutil
from datetime import datetime

backup_ts = datetime.now().strftime("%Y%m%d_%H%M%S")

source_folder = "/lakehouse/default/Files/backup/lakehouse_full_table_backup_20260601_021744"

zip_file = f"/lakehouse/default/Files/backup/lakehouse_backup_{backup_ts}"

shutil.make_archive(
    zip_file,
    'zip',
    source_folder
)

print("ZIP created:", zip_file + ".zip")

ZIP created: /lakehouse/default/Files/backup/lakehouse_backup_20260601_021958.zip
